# ***Simple Gen AI APP Using Langchain***

### ***Step 1: Environment Setup & API Key Configuration***

In [12]:
import os
from dotenv import load_dotenv

In [13]:
# Load environment variables from a .env file
load_dotenv()

True

In [14]:
# Set up API keys from environment variables
os.environ['HUGGINGFACEHUB_API_TOKEN'] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
# Langsmith Tracking (for monitoring and debugging Langchain applications)
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

### ***Step 2: Web Scraping (Data Ingestion)***

In [15]:
# Data Ingestion - Load content from a website
from langchain_community.document_loaders import WebBaseLoader

In [16]:
# Initialize the web scraper with a target URL
loader = WebBaseLoader("https://python.langchain.com/docs/introduction/")

In [17]:
# This will return a list of Document objects
docs=loader.load()

In [18]:
docs

[Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='\n\n\n\n\nIntroduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain\n\n\n\n\n\n\nSkip to main contentJoin us at  Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in San Francisco!IntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1\uf8ffüí¨SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a simple LLM application with chat models and prompt templatesBuild a ChatbotBuild a Retrieval Augmented Generation (RAG) App: Part 2Build an Extraction ChainBuild an AgentTaggingBuild a Retrieval Augmented Generation (RAG) App: Part 1Build a semantic search engineBuild a Question/Answering system

### ***Step 3: Splitting Documents into Chunks***

In [19]:
# Split documents into smaller chunks for better processing
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [20]:
# Initialize the text splitter with chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [21]:
# Split the documents into smaller parts
documents = text_splitter.split_documents(docs)

In [22]:
documents

[Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain'),
 Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='Skip to main contentJoin us at  Interrupt: The Agent AI Conference by LangChain on May 13 & 14 in San Francisco!IntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1\uf8ffüí¨SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a simple LLM application with cha

### ***Step 4: Converting Text into Embeddings***

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings

In [24]:
# Use a small, efficient embedding model to convert text into vectors
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ***Step 5: Storing Embeddings in a FAISS Vector Database***

In [25]:
from langchain_community.vectorstores import FAISS

In [26]:
# Create a FAISS vector store database from the document embeddings
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [27]:
vectorstoredb

### ***Step 6: Querying the FAISS Vector Store***

In [28]:
# Test vector search by querying similar content from the vector database
query = "LangSmith has two usage limits: total traces and extended"

In [29]:
result = vectorstoredb.similarity_search(query)

In [31]:
# Print the most relevant document
print(result[1].page_content)

LangChain is a framework for developing applications powered by large language models (LLMs).
LangChain simplifies every stage of the LLM application lifecycle:


### ***Step 7: Loading an LLM from Hugging Face***

In [37]:
from langchain_community.llms import HuggingFaceHub

In [38]:
# Load a lightweight, CPU-friendly LLM model from Hugging Face Hub
llm = HuggingFaceHub(repo_id="google/gemma-2b")  # Smallest & most efficient model

C:\Users\emon1\AppData\Local\Temp\ipykernel_12796\3403651261.py:2: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(repo_id="google/gemma-2b")  # Smallest & most efficient model


### ***Step 8: Creating a Document Chain for Answer Generation***

In [39]:
# Retrieval Chain and Document Chain Setup
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [40]:
# Define a prompt template that instructs the LLM to answer based on context
prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context:
<context>
{context}
</context>
""")

In [41]:
# Create a document chain that processes retrieved documents with the LLM
document_chain = create_stuff_documents_chain(llm, prompt)

### ***Step 9: Testing the Document Chain***

In [42]:
from langchain_core.documents import Document

In [43]:
# Test the document chain by providing some context manually
response = document_chain.invoke({
    "input": "LangSmith has two usage limits: total traces and extended",
    "context": [Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph.")]
})

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


In [44]:
# Print the response generated by the LLM
print(response)

Human: 
Answer the following question based only on the provided context:
<context>
LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph.
</context>
Question: What are the two usage limits of LangSmith?
Answer: The two usage limits of LangSmith are total traces and extended traces.


### ***Step 10: Creating a Retrieval Chain***

In [45]:
# Convert the FAISS vector store into a retriever
retriever = vectorstoredb.as_retriever()

In [46]:
from langchain.chains import create_retrieval_chain

In [47]:
# Create a retrieval chain that first retrieves relevant documents and then generates answers
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [50]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001BD1A91FE90>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
            |

### ***Step 11: Querying the System***

In [48]:
# Query the retrieval chain to get an AI-generated response
response = retrieval_chain.invoke({"input": "LangSmith has two usage limits: total traces and extended"})

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


In [51]:
response

{'input': 'LangSmith has two usage limits: total traces and extended',
 'context': [Document(id='7850240a-b43b-4d56-8933-2ae7b9048922', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain'),
  Document(id='c0d1dc2a-c9b9-4ca4-8622-f3a6666e98b0', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='LangChain is a framework for developing applications powered by large language models (LLMs).\nLangChain simplifies every stage of the LLM application lifecycle:'),
  Document(id='6b245578-0083-42bb-a07e-ca4c83

In [54]:
response['context']

[Document(id='7850240a-b43b-4d56-8933-2ae7b9048922', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain'),
 Document(id='c0d1dc2a-c9b9-4ca4-8622-f3a6666e98b0', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | \uf8ffü¶úÔ∏è\uf8ffüîó LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='LangChain is a framework for developing applications powered by large language models (LLMs).\nLangChain simplifies every stage of the LLM application lifecycle:'),
 Document(id='6b245578-0083-42bb-a07e-ca4c83594ced', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'titl

In [49]:
# Print the final answer
print(response['answer'])

Human: 
Answer the following question based only on the provided context:
<context>
Introduction | ü¶úÔ∏èüîó LangChain

LangChain is a framework for developing applications powered by large language models (LLMs).
LangChain simplifies every stage of the LLM application lifecycle:

For a deeper dive into LangGraph concepts, check out this page.
Integrations‚Äã
LangChain is part of a rich ecosystem of tools that integrate with our framework and build on top of it.
If you're looking to get up and running quickly with chat models, vector stores,
or other LangChain components from a specific provider, check out our growing list of integrations.
API reference‚Äã
Head to the reference section for full documentation of all classes and methods in the LangChain Python packages.
Ecosystem‚Äã
ü¶úüõ†Ô∏è LangSmith‚Äã
Trace and evaluate your language model applications and intelligent agents to help you move from prototype to production.
ü¶úüï∏Ô∏è LangGraph‚Äã
Build stateful, multi-actor applic